In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

d:\projects\GENAi\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\mkdog\AppData\Local\Temp\ipykernel_19156\125792775.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7624.53it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can b

In [3]:
from langchain_community.vectorstores import InMemoryVectorStore

In [4]:
loader = PyPDFLoader("../Data/mayank.pdf")

In [5]:
docs = loader.load()
len(docs)

43

In [6]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1200,chunk_overlap=450)
splitted_docs = splitter.split_documents(docs)
len(splitted_docs)

62

In [7]:
vector_store = InMemoryVectorStore.from_documents(
    documents =splitted_docs,
    embedding=embeddings,
)

In [8]:
res = vector_store.similarity_search("author name")

In [9]:
len(res)

4

In [10]:
res[0].page_content

'CERTIFICATE OF ORIGINALITY \n \nI hereby certify that the   work which   is being presented in the B.Tech. Industrial \nProject Report entitled “FraudShield – Smart Fraud Detection”, in partial fulfillment \nof the requirements for the award of the Bachelor of Technology in Computer science \nEngineering, and submitted to the Department of Computer Science  Engineering of \nAarni University, Indora Himachal Pradesh, is an authentic record of my own work \ncarried out during the period from June 2025 to July 2025 under the supervision of Er. \nTushar Devkaran. \nSignature \n                                                                                                                        Mayank Kumar  \nABCS0008A/22 \nThis is to certify that the above statement made by the candidate is correct to the best of my \nknowledge. \n \nDate:    \nSignature of Supervisor  \nER TUSHAR DEVKARAN \n(Project Manager)'

In [11]:
from langchain.tools import tool

In [35]:
@tool
def retriever_tool(query:str):
    """
    This tool can help you to retrieve the relevant data of the pdf documents.
    
    """
    print("Tool called :",query)
    res_new =vector_store.similarity_search(query=query,k=4)
    context = ""

    for doc in res_new:
        context = doc.page_content + "\n\n"

    return context

In [13]:
retriever_tool.invoke("Mayank")

'ANALYSIS \n \n5.1 STRENGTHS \nFraudShield provides multiple technical and operational strengths that make it a \nreliable system for fraud detection: \n• High Accuracy : Leveraging the Random Forest classifier results in \nexcellent performance in classifying fraudulent vs. legitimate transactions. \n• Robustness to Imbalanced Data: The integration of SMOTE (Synthetic \nMinority Oversampling Technique) improves the model’s ability to detect \nminority class transactions (fraud cases). \n• Open-source Environment: Built using free and open -source tools like \nPython, Scikit -learn, and Pandas, minimizing cost and improving \naccessibility. \n• Scalability: The model and architecture are modular, allowing future \nexpansion or integration into a full-scale real-time system. \n• Visualization Support : Libraries like Matplotlib and Seaborn offer \neffective insights into data and results, useful for analysis and \npresentations. \n• Ease of Implementation: The system is easy to set up o

In [30]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [16]:
system_prompt = """
    You are a helpful assistant that answers question using retrieved context,
    always use "retriever_tool" for a question requiring external knowlwedge

"""

In [17]:
from langchain.agents import create_agent

In [36]:
agent = create_agent(
    model= llm,
    tools=[retriever_tool],
    system_prompt=system_prompt
)

In [38]:
query = "Processor requirements , and Ram requirements"
res = agent.invoke({"messages":[{"role":"user","content":query}]})
result = res["messages"][-1].content
result

Tool called : Processor requirements and RAM requirements


[{'type': 'text',
  'text': 'The project requires a mid-range laptop or desktop with an Intel i5/Ryzen 5 processor and 8–16 GB RAM.',
  'extras': {'signature': 'CvkDAb4+9vuZ8t6wMiI6nfKNv7Z2smZr/jhboU8T8ENsb9/w/tNUQBTRrA8mV4VX+7Uteub2SdUnXX4ojNYrDR942RIcFOz0TtjoCMX7W5UFJiRoIybyItRJRFE1OO/I0lpT9Eo/LjvWh7AREbG8TxYw+5TZySw5O8FfOGgKyU7RwjAFe/cdaghOZ3aFs8Cq+QK6IPV5VCcJfidn862C/NtxdRzHZIbIwWo+e5kkASgIMfKSNmJQUn4HJUy6JQbaTUIFfi93xpOySkhFXtuM071plfeoj0ZPpRmNL+YvMGMKFy6cV5ENEd8lB+rDotZPtw1yrSHlYpxtAabPg3+XD6sz4fLcv3m/Jx5qJl5CG4Vfq3raQ3GPTIFyCUVybQl08jK8PmcrGpvQzYkV0ZMWyQsBBzv7Hiv6VFyt4hTTgA9XVCPg7ZxZSIjICJsy44tXXcC2/orrFt4eW9upMUIyFX7IuUyYAgWsTXLkpKj3AlWxRcQmcszx4UMm1Jgp331r4hugb5d/jipPtBzQBkdcqe4crMi2tMkD63eOcH9ePkgC/jSwCJrHgvbAihVPlkZcLZkCtXndpaXk+/R+FQUHDg+wf/w6MZrhm+k2+BgTGJfxa+AjjHzSwK2rz1luSQcjtRERglA2Rg2V7qZFvxHrI1LeWaB5v5sNDdcIOw=='}}]